### LLM Gateways - With LiteLLM + LangChain

#### What is an LLM Gateway?
Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq

##### Without a Gateway (The Pain):
* Different SDKs and APIs for every provider
* No fallback if one provider goes down
* No central place to track costs
* Hard to switch models without rewriting code
* No caching → paying twice for the same query

##### With a Gateway (The Joy):
* One unified API for 100+ providers
* Automatic fallbacks if a provider fails
* Centralized logging, cost tracking, rate limiting
* Swap models with a config change, no code rewrite
* Cache repeated queries → save money

Installation & Setup:
We'll use:
* LiteLLM → the most popular open-source LLM gateway (supports 100+ providers)
* LangChain → for building agentic workflows on top of the gateway
* python-dotenv → for managing API keys

In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

# Keep the recording clean - suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

print("OpenAI key loaded: ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Gemini key loaded: ", "✅" if os.getenv("GEMINI_API_KEY") else "❌")
print("Groq key loaded: ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:  ✅
Gemini key loaded:  ✅
Groq key loaded:  ✅


#### The Simplest LiteLLM Example - Unified API
The biggest pain point: every provider has a different SDK.

LiteLLM gives you one function —> completion() —> that works with all of them. Look at how clean this is:

In [5]:
from litellm import completion

# Call Gemini
response_gemini = completion(
    model = "gemini/gemini-3.6-flash",
    messages = [{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("Gemini: ", response_gemini.choices[0].message.content)


# Call Groq (super fast inference)
response_groq = completion(
    model = "groq/openai/gpt-oss-120b",
    messages = [{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("Groq: ", response_groq.choices[0].message.content)

Gemini:  **Retrieval-Augmented Generation (RAG)** is an AI technique that enhances language models by fetching relevant information from external databases before generating a response, ensuring answers are accurate, up-to-date, and context-specific.
Groq:  Retrieval‑Augmented Generation (RAG) is a technique that combines a language model with a searchable external knowledge base, retrieving relevant documents at inference time and feeding them into the model so it can generate answers grounded in up‑to‑date, factual information.


**Notice:** Same code, three different providers. This alone is huge - you can switch providers with a string change.

But a real LLM Gateway does much more. Let's build those features one by one.

In [6]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("OpenAI", "gpt-4o-mini"),
    ("Groq", "groq/openai/gpt-oss-120b"),
    ("Anthropic", "claude-3-5-haiku-20241022"),
    ("Gemini", "gemini/gemini-3.6-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

OpenAI         : ❌ RateLimitError
Groq           : Retrieval‑Augmented Generation (RAG) is a technique that integrates a language m
Anthropic      : ❌ BadRequestError
Gemini         : **Retrieval-Augmented Generation (RAG)** is an AI technique that enhances langua


#### Automatic Fallbacks - When a Model Goes Down
**Real story:** OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we **automatically fall back to another**. Production apps must have this.

In [7]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model = "gemini/gemini-1.5-flash",
    messages = [{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks = [
        "gpt-4o-mini",
        "groq/openai/gpt-oss-120b"
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

16:30:00 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "d:\Coding\Agentic AI\01. LangChain\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2842, in async_completion
    response: Final = await client.post(
                      ^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "d:\Coding\Agentic AI\01. LangChain\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 338, in async_wrapper
    result: Final = await func(*args, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Coding\Ag

Response: **LLM Gateway – A Quick Overview**

| Aspect | Description |
|--------|-------------|
| **What it is** | A **gateway** (or **proxy**) that sits between applications (or end‑users) and one or more Larg ...

Which model actually answered? openai/gpt-oss-120b


#### Cost Tracking - Know Where Your Money Goes
LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills.

In [8]:
from litellm import completion, completion_cost

response = completion(
    model = "groq/openai/gpt-oss-120b",
    messages = [{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response = response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Exception: This model isn't mapped yet. model=openai/gpt-oss-120b, custom_llm_provider=openai. Add it here - https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json.

#### Caching - Don't Pay Twice for the Same Question
If 100 users ask "What is RAG?", you don't need to call the LLM 100 times.

Enable in-memory caching with one line:

In [9]:
import litellm

# Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("LiteLLM state reset - ready for clean caching demo")

LiteLLM state reset - ready for clean caching demo


In [10]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line."

# First call - actually hits LLM
start = time.time()
r1 = completion(
    model = "groq/openai/gpt-oss-120b",
    messages = [{"role": "user", "content": prompt}],
    caching = True
)
t1 = time.time() - start
print(f"First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model = "groq/openai/gpt-oss-120b",
    messages = [{"role": "user", "content": prompt}],
    caching = True
)
t2 = time.time() - start
print(f"Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\nSpeedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

First call (API):   0.80s — LLM stands for **Large Language Model**.


ModuleNotFoundError: No module named 'soundfile'

#### Smart Routing - The Right Model for the Right Job
Why use one model for everything?
* Coding tasks → Claude Sonnet
* Cheap summaries → GPT-4o-mini
* Super fast replies → Groq Llama
* Complex reasoning → Claude Opus

Use LiteLLM's Router to define routing rules:

In [ ]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model": "gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model = "fast-cheap",
    messages = [{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model = "smart-coding",
    messages = [{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\nSmart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])

#### Load Balancing Across Multiple API Keys
Hit rate limits on one OpenAI key? Add more keys to the same alias - the router load-balances automatically.

In [ ]:
from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models - all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
        "model_info": {"id": "openai-gpt4o"}
    },
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list = model_list,
    routing_strategy = "simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

##### Strategy 1: least-busy
**The "Express Checkout" Pattern**

The idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

In [ ]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {"id": "OpenAI"}
    },
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info": {"id": "Groq"}
     },
]

router = Router(
    model_list = model_list,
    routing_strategy = "least-busy" # Defining our strategy of Least Busy
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model = "chat",
        messages = [{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens = 5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\nDistribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

##### Strategy 2: latency-based-routing
**The "Always Pick the Fastest" Pattern**

The idea: The router measures the response time of each deployment over recent calls and sends new requests to whichever has been fastest. Speed wins.

In [ ]:
import os
from litellm import Router
import time

model_list = [
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {"id": "OpenAI GPT-4o-mini"}
     },
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info": {"id": "Groq Llama-3.3"}
     },
    
]

router = Router(
    model_list = model_list,
    routing_strategy = "latency-based-routing" # Defining our strategy Least Latency
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model = "chat",
        messages = [{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens = 5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

##### Strategy 4: cost-based-routing
**The "Always Cheapest" Pattern**

The idea: Pick the deployment that costs the least per token right now. Beautiful for cost-sensitive apps.

In [ ]:
import os
from litellm import Router

# Different providers with very different price points
model_list = [
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "gpt-4o",             # ~$2.50/M input tokens
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {"id": "GPT-4o (premium)"}
    },
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "gpt-4o-mini",        # ~$0.15/M input tokens
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {"id": "GPT-4o-mini (cheap)"}
     },
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",   # ~$0.05/M
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info": {"id": "Groq Llama (cheapest)"}
     },
]

router = Router(
    model_list = model_list,
    routing_strategy = "simple-shuffle" # Defining our strategy of Least Cost
)

for i in range(5):
    r = router.completion(
        model = "chat",
        messages = [{"role": "user", "content": "Hi"}],
        max_tokens = 10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")